# Notebook A v4.3: Research Actions + Signal Weights

**Run order:** First. Before Notebook B v4.4 and Notebook D.

## v4.3 changes vs v4.2

### Prerequisite
Notebook 08 must have been run with `is_blood_relative` added to `gold_ancestral_proximity`,
and Notebook B v4.4 must have been run to expose `is_blood_relative` as a context column
in `gold_research_person_signals`.

### New applicability flag column
- `requires_blood_relative` (column 22) — signal only applicable if `is_blood_relative = TRUE`.
  Suppresses completeness and evidence signals for spouses and in-laws, whose only
  meaningful research goal is basic vitals.

### Signals updated with requires_blood_relative = TRUE
The following signals are now suppressed for non-blood relatives (spouses and in-laws).
Rationale: for spouses we only need name, birth, death — not census coverage, occupation,
residence, late life records, or any evidence-quality signals.

| Signal | Reason |
|---|---|
| `SIGNAL_MISSING_CENSUS_COVERAGE` | Census research not a goal for spouses |
| `SIGNAL_DOCS_NOT_TRANSCRIBED` | Document pipeline not applicable for spouses |
| `SIGNAL_LATE_LIFE_GAP` | Late-life records not a goal for spouses |
| `SIGNAL_EARLY_LIFE_ONLY` | Adult record gap not a priority for spouses |
| `SIGNAL_CHILD_GAPS` | Child gap research not a goal for spouses |
| `SIGNAL_MISSING_OCCUPATION` | Occupation research not a goal for spouses |
| `SIGNAL_MISSING_BURIAL` | Burial research not a goal for spouses |
| `SIGNAL_NO_RESIDENCE` | Residence research not a goal for spouses |
| `SIGNAL_NO_DOCUMENTS_AT_ALL` | Document coverage not a goal for spouses |
| `SIGNAL_FACT_CONFLICT` | Transcript pipeline only relevant for blood relatives |
| `SIGNAL_TRANSCRIPT_ONLY_FACTS` | Transcript pipeline only relevant for blood relatives |
| `SIGNAL_VERY_LOW_EVIDENCE_DENSITY` | Evidence quality not a goal for spouses |
| `SIGNAL_LOW_EVIDENCE_DENSITY` | Evidence quality not a goal for spouses |
| `SIGNAL_SINGLE_SOURCE_DEPENDENCE` | Source corroboration not a goal for spouses |
| `SIGNAL_UNSOURCED_FAMILY_EVENTS` | Sourcing family events not a goal for spouses |
| `SIGNAL_IMPRECISE_DATES` | Date precision not a goal for spouses |
| `SIGNAL_IMPRECISE_PLACES` | Place precision not a goal for spouses |
| `SIGNAL_UNCOVERED_SOURCES` | Source coverage not a goal for spouses |

Signals intentionally left without requires_blood_relative (remain applicable for all):
- `SIGNAL_NO_BIRTH_RECORDED` — vitals are needed for spouses
- `SIGNAL_NO_DEATH_RECORDED` — vitals are needed for spouses
- `SIGNAL_NO_MARRIAGES` — needed for spouses (they may have remarried)
- `SIGNAL_NO_CHILDREN` — proximity guard (<=1) already limits this appropriately
- `SIGNAL_MISSING_PARENT` — useful to know if a spouse's parents are known
- `SIGNAL_INCOMPLETE_NAME` — always applicable
- All narrative signals — applies_always, not relevant to suppress
- All DNA signals — tag-based applicability already handles this correctly
- `SIGNAL_UNCONFIRMED_MILITARY` — note: buggy (fires too broadly), fix in Notebook B pending

### Live table sync
This notebook is synced to the live `ref_signal_weights` table as of May 2026, including:
- `outcome_label` column added in Session 47
- Revised base scores (NO_BIRTH_RECORDED 30, NO_DOCUMENTS_AT_ALL 40,
  VERY_LOW_EVIDENCE_DENSITY 30, SINGLE_SOURCE_DEPENDENCE 25,
  UNSOURCED_FAMILY_EVENTS 25, TRANSCRIPT_ONLY_FACTS 20, MISSING_BURIAL 8)
- UNCOVERED_SOURCES moved to evidence dimension
- DNA signals (CITATION_MISSING, PATH_INCOMPLETE) in evidence dimension
- UNSOURCED_FAMILY_EVENTS has applies_always = TRUE (not requires_family_events)

## v4.2 changes vs v4.1

DNA signal support added:
- Three new applicability flag columns: `requires_dna_match_tag`, `requires_dna_ancestor_tag`,
  `requires_generation_lte`
- New action: `INVESTIGATE_DNA_MATCH`
- Four new signals: DNA_CITATION_MISSING, DNA_PATH_INCOMPLETE,
  NO_DNA_CORROBORATION, COMMON_ANCESTOR_UNVERIFIED

## v4.1 changes vs v4

Action code cleanup — CLUSTER_SOURCES and REVIEW_EXISTING_SOURCES retired;
replaced by RECONCILE_TRANSCRIPT_FACTS and ORDER_BMD_CERTIFICATE.

## Applicability flag columns (all 22)

| # | Column | Type | Meaning |
|---|---|---|---|
| 01 | `applies_always` | BOOLEAN | In denominator for all people (overrides other flags if TRUE) |
| 02 | `min_birth_year` | INT | Only applicable if birth_year >= N |
| 03 | `max_birth_year` | INT | Only applicable if birth_year <= N |
| 04 | `sex_restriction` | STRING | 'M' or 'F'; NULL = any sex |
| 05 | `requires_expected_to_die` | BOOLEAN | birth_year <= 1930 AND expected_end_year < current_year |
| 06 | `requires_lifespan_gte` | INT | effective_span_years >= N |
| 07 | `requires_survived_past_40` | BOOLEAN | expected_end_year >= birth_year + 40 |
| 08 | `requires_working_age` | BOOLEAN | expected_end_year >= birth_year + 18 |
| 09 | `requires_census_years` | BOOLEAN | At least one expected census year > 0 |
| 10 | `requires_post_1837_event` | BOOLEAN | birth/death/marriage >= 1837 |
| 11 | `requires_has_document` | BOOLEAN | Has >= 1 matched file in silver_document_person |
| 12 | `requires_transcript` | BOOLEAN | Has >= 1 transcribed non-BMDIndex document |
| 13 | `requires_citations` | BOOLEAN | Has >= 1 citation in gold_source_coverage |
| 14 | `requires_family_events` | BOOLEAN | Has marriages or child births |
| 15 | `requires_proximity_lte` | INT | proximity <= N |
| 16 | `requires_total_facts_gte` | INT | total_facts >= N |
| 17 | `requires_death_recorded` | BOOLEAN | death_year IS NOT NULL |
| 18 | `not_young_death` | BOOLEAN | Suppress if effective_span_years BETWEEN 16 AND 40 |
| 19 | `requires_dna_match_tag` | BOOLEAN | Person has DNA Match tag (@T10@) |
| 20 | `requires_dna_ancestor_tag` | BOOLEAN | Person has Common DNA Ancestor tag (@T5@) |
| 21 | `requires_generation_lte` | INT | Generation depth <= N |
| 22 | `requires_blood_relative` | BOOLEAN | **NEW v4.3** — is_blood_relative = TRUE (suppresses signals for spouses/in-laws) |


In [0]:
%sql
CREATE OR REPLACE TABLE genealogy.gold_research_action (
  action_code     STRING,
  action_label    STRING,
  action_category STRING,
  default_effort  INT
);

INSERT INTO genealogy.gold_research_action VALUES
('RESOLVE_NAME_VARIANTS',        'Resolve name variants / aliases',                     'identity',  2),
('RESOLVE_CONFLICTS',            'Investigate and resolve transcript vs tree conflicts', 'evidence',  2),
('RECONCILE_TRANSCRIPT_FACTS',   'Update tree from transcript facts',                   'evidence',  2),
('ORDER_BMD_CERTIFICATE',        'Order BMD certificate',                               'evidence',  2),
('SEARCH_BIRTH_RECORDS',         'Search for birth/baptism records',                    'lifecycle', 1),
('SEARCH_DEATH_RECORDS',         'Search for death/burial records',                     'lifecycle', 1),
('SEARCH_BURIAL_RECORDS',        'Search for burial/cremation records',                 'lifecycle', 1),
('SEARCH_MARRIAGE_RECORDS',      'Search for marriage records',                         'lifecycle', 1),
('SEARCH_CENSUS',                'Search census / population records',                  'lifecycle', 1),
('DOWNLOAD_DOCUMENTS',           'Download cited documents from Ancestry/archive',      'evidence',  2),
('TRANSCRIBE_DOCUMENTS',         'Run OCR transcription pipeline on downloaded docs',   'evidence',  1),
('TRACE_PARENTS',                'Identify or verify parents',                          'family',    2),
('TRACE_CHILDREN_FORWARD',       'Trace children forward',                              'family',    2),
('VERIFY_FAMILY_EVENTS',         'Source family events (marriage, children)',            'family',    1),
('INVESTIGATE_MIGRATION',        'Investigate migration and movement',                  'narrative', 2),
('SEARCH_MILITARY_RECORDS',      'Search military service records',                     'narrative', 2),
('INVESTIGATE_MILITARY',         'Investigate and record confirmed military service',   'narrative', 3),
('INVESTIGATE_OCCUPATION',       'Research occupational history and records',           'narrative', 2),
('INVESTIGATE_LATER_LIFE',       'Search for late-life events and records',             'narrative', 2),
('INVESTIGATE_DNA_MATCH',        'Research and link DNA match in tree',                 'dna',       2);


In [0]:
%sql
CREATE OR REPLACE TABLE genealogy.ref_intent_category_weights (
  intent   STRING,
  category STRING,
  weight   DOUBLE
);

INSERT OVERWRITE genealogy.ref_intent_category_weights VALUES
  ('integrity', 'evidence',     0.60),
  ('integrity', 'completeness', 0.40),
  ('narrative', 'texture',      0.45),
  ('narrative', 'family',       0.30),
  ('narrative', 'context',      0.25);


In [0]:
%sql
-- ============================================================
-- CELL 3: ref_signal_weights (v4.3)
--
-- v4.3 changes vs v4.2:
--   ONE NEW flag column added (column 22):
--     requires_blood_relative — only applicable for blood relatives
--       (is_blood_relative = TRUE in gold_research_person_signals).
--       Suppresses signals for spouses and in-laws whose only meaningful
--       research goal is basic vitals (birth, death, name).
--       Requires Notebook B v4.4 and Notebook 08 to have been run first.
--
--   SIGNALS UPDATED: requires_blood_relative = TRUE set on 18 signals.
--   See markdown header for full list and rationale.
--
-- SYNCED TO LIVE TABLE (May 2026):
--   - outcome_label column present (added Session 47)
--   - base_score values match live: NO_BIRTH_RECORDED=30, NO_DOCUMENTS_AT_ALL=40,
--     VERY_LOW_EVIDENCE_DENSITY=30, SINGLE_SOURCE_DEPENDENCE=25,
--     UNSOURCED_FAMILY_EVENTS=25, TRANSCRIPT_ONLY_FACTS=20, MISSING_BURIAL=8
--   - UNCOVERED_SOURCES in evidence dimension
--   - DNA signals CITATION_MISSING and PATH_INCOMPLETE in evidence dimension
--   - UNSOURCED_FAMILY_EVENTS: applies_always=TRUE (not requires_family_events)
--
-- NULL in a flag column means restriction does not apply.
-- applies_always = TRUE means: ignore all other flags, always in denominator.
-- Note: requires_blood_relative is evaluated AFTER applies_always — a signal
-- with applies_always=TRUE will still be suppressed for non-blood relatives
-- if requires_blood_relative=TRUE is set. Notebook D must implement this.
-- ============================================================

CREATE OR REPLACE TABLE genealogy.ref_signal_weights (
  signal_code               STRING,
  category                  STRING,
  intent                    STRING,
  base_score                INT,
  reason_label              STRING,
  rationale                 STRING,
  dimension                 STRING,
  -- applicability flags
  applies_always            BOOLEAN, --01
  min_birth_year            INT,     --02
  max_birth_year            INT,     --03
  sex_restriction           STRING,  --04
  requires_expected_to_die  BOOLEAN, --05
  requires_lifespan_gte     INT,     --06
  requires_survived_past_40 BOOLEAN, --07
  requires_working_age      BOOLEAN, --08
  requires_census_years     BOOLEAN, --09
  requires_post_1837_event  BOOLEAN, --10
  requires_has_document     BOOLEAN, --11
  requires_transcript       BOOLEAN, --12
  requires_citations        BOOLEAN, --13
  requires_family_events    BOOLEAN, --14
  requires_proximity_lte    INT,     --15
  requires_total_facts_gte  INT,     --16
  requires_death_recorded   BOOLEAN, --17
  not_young_death           BOOLEAN, --18
  requires_dna_match_tag    BOOLEAN, --19
  requires_dna_ancestor_tag BOOLEAN, --20
  requires_generation_lte   INT,     --21
  -- NEW in v4.3
  requires_blood_relative   BOOLEAN, --22
  -- outcome_label: imperative action-framed label shown in This Week action list
  outcome_label             STRING
);

INSERT OVERWRITE genealogy.ref_signal_weights VALUES

-- ================================================================
-- COMPLETENESS signals
-- Column order: signal_code, category, intent, base_score, reason_label, rationale, dimension,
--   01:applies_always, 02:min_birth_year, 03:max_birth_year, 04:sex_restriction,
--   05:requires_expected_to_die, 06:requires_lifespan_gte, 07:requires_survived_past_40,
--   08:requires_working_age, 09:requires_census_years, 10:requires_post_1837_event,
--   11:requires_has_document, 12:requires_transcript, 13:requires_citations,
--   14:requires_family_events, 15:requires_proximity_lte, 16:requires_total_facts_gte,
--   17:requires_death_recorded, 18:not_young_death,
--   19:requires_dna_match_tag, 20:requires_dna_ancestor_tag, 21:requires_generation_lte,
--   22:requires_blood_relative, outcome_label
-- ================================================================

-- NO_BIRTH_RECORDED: always applicable including spouses (vitals needed)
('SIGNAL_NO_BIRTH_RECORDED', 'completeness', 'integrity', 30,
 'birth not recorded', NULL, 'completeness',
 TRUE,  NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
 'Confirm birth'),

-- NO_DEATH_RECORDED: always applicable including spouses (vitals needed)
('SIGNAL_NO_DEATH_RECORDED', 'completeness', 'integrity', 20,
 'death not recorded', NULL, 'completeness',
 FALSE, NULL, 1930, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
 'Confirm death'),

-- NO_MARRIAGES: not blood-restricted — spouses may have remarried after our ancestor died
('SIGNAL_NO_MARRIAGES', 'completeness', 'integrity', 15,
 'no marriage recorded', 'age-guarded: suppressed for young deaths', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL,
 'Find marriage record'),

-- NO_CHILDREN: proximity guard (<=1) already limits this; not blood-restricted additionally
('SIGNAL_NO_CHILDREN', 'completeness', 'integrity', 10,
 'no children recorded', 'proximity guard: proximity <= 1 only', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 1, NULL, NULL, TRUE, NULL, NULL, NULL, NULL,
 'Find children'),

-- MISSING_PARENT: not blood-restricted — knowing a spouse's parents can be useful
('SIGNAL_MISSING_PARENT', 'completeness', 'integrity', 25,
 'missing parent', NULL, 'completeness',
 TRUE,  NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
 'Find parents'),

-- MISSING_CENSUS_COVERAGE: blood relatives only — census research not a goal for spouses
('SIGNAL_MISSING_CENSUS_COVERAGE', 'completeness', 'integrity', 20,
 'missing expected census', 'person was alive during census year but no RESI event present', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find missing censuses'),

-- DOCS_NOT_TRANSCRIBED: blood relatives only — document pipeline not applicable for spouses
('SIGNAL_DOCS_NOT_TRANSCRIBED', 'completeness', 'integrity', 15,
 'documents not transcribed', 'evidence downloaded but not yet verified via OCR', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Transcribe documents'),

-- LATE_LIFE_GAP: blood relatives only — late-life research not a goal for spouses
('SIGNAL_LATE_LIFE_GAP', 'completeness', 'integrity', 15,
 'late-life records gap', 'no events recorded after age 40 despite expected survival', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Fill late life gap'),

-- EARLY_LIFE_ONLY: blood relatives only — adult record gap not a priority for spouses
('SIGNAL_EARLY_LIFE_ONLY', 'completeness', 'integrity', 10,
 'early-life records only', 'classic birth+parents-only profile, no adult records', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find adult records'),

-- CHILD_GAPS: blood relatives only — child gap research not a goal for spouses
('SIGNAL_CHILD_GAPS', 'completeness', 'integrity', 10,
 'gaps between child births', 'unusually long inter-birth gap or marriage with no children', 'completeness',
 TRUE,  NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find gaps between children'),

-- UNCONFIRMED_MILITARY: not blood-restricted (spouses could also have served)
-- NOTE: buggy — fires on all ~1,996 people born 1867-1929 with no military event.
-- Fix pending in Notebook B: require num_military > 0 AND NOT has_military_record.
('SIGNAL_UNCONFIRMED_MILITARY', 'completeness', 'integrity', 15,
 'military service unconfirmed', 'born 1867-1928 — WWI/WWII service neither confirmed nor denied', 'completeness',
 FALSE, 1867, 1929, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
 'Confirm military service'),

-- MISSING_OCCUPATION: blood relatives only — occupation research not a goal for spouses
('SIGNAL_MISSING_OCCUPATION', 'completeness', 'integrity', 10,
 'no occupation recorded', 'male with working-age lifespan and zero occupation events recorded', 'completeness',
 FALSE, NULL, NULL, 'M', NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find occupation records'),

-- MISSING_BURIAL: blood relatives only — burial research not a goal for spouses
('SIGNAL_MISSING_BURIAL', 'completeness', 'integrity', 8,
 'burial not recorded', 'post-1837 death with no burial or cremation event', 'completeness',
 TRUE,  NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find burial record'),

-- NO_RESIDENCE: blood relatives only — residence research not a goal for spouses
('SIGNAL_NO_RESIDENCE', 'completeness', 'integrity', 10,
 'no residence events', 'lifespan >= 20 years with zero residence/census events', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, 20, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find residence records'),

-- POSSIBLE_MARRIAGE: context hint only (base_score=0)
('SIGNAL_POSSIBLE_MARRIAGE', 'completeness', 'integrity', 0,
 'possible unrecorded marriage', 'female in 1939 register with no marriage recorded', 'completeness',
 FALSE, NULL, NULL, 'F', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
 'Investigate possible marriage'),

-- POSSIBLE_CHILDREN: context hint only (base_score=0)
('SIGNAL_POSSIBLE_CHILDREN', 'completeness', 'integrity', 0,
 'possible unrecorded children', 'female in 1911 census born <=1895', 'completeness',
 FALSE, NULL, 1895, 'F', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
 'Investigate possible children'),

-- POSSIBLE_RESIDENCE: context hint only (base_score=0)
('SIGNAL_POSSIBLE_RESIDENCE', 'completeness', 'integrity', 0,
 'residence records likely findable', NULL, 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
 'Investigate possible residence'),

-- ================================================================
-- EVIDENCE signals
-- ================================================================

-- NO_DOCUMENTS_AT_ALL: blood relatives only — document coverage not a goal for spouses
('SIGNAL_NO_DOCUMENTS_AT_ALL', 'evidence', 'integrity', 40,
 'no documents filed', 'zero matched files — profile entirely uncorroborated by downloaded evidence', 'evidence',
 FALSE, 1600, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find any documents'),

-- FACT_CONFLICT: blood relatives only — transcript pipeline only relevant for blood relatives
('SIGNAL_FACT_CONFLICT', 'evidence', 'integrity', 30,
 'transcript conflicts tree', 'document contradicts recorded fact', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Resolve fact conflicts'),

-- VERY_LOW_EVIDENCE_DENSITY: blood relatives only — evidence quality not a goal for spouses
('SIGNAL_VERY_LOW_EVIDENCE_DENSITY', 'evidence', 'integrity', 30,
 'very low source density (mostly unsourced)', 'avg sources per fact < 0.3 — profile largely unsourced', 'evidence',
 TRUE,  NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Add more sources'),

-- DNA_PATH_INCOMPLETE: tag-based applicability handles blood/non-blood correctly
('SIGNAL_DNA_PATH_INCOMPLETE', 'evidence', 'integrity', 25,
 'DNA path tags incomplete', 'Path to researcher has missing DNA Connection or Common Ancestor tags', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL,
 'Complete DNA path'),

-- NO_DNA_CORROBORATION: tag-based + proximity guard handles correctly
('SIGNAL_NO_DNA_CORROBORATION', 'evidence', 'integrity', 25,
 'ancestor not DNA-corroborated', 'Direct ancestor gen 1-6 with no Common DNA Ancestor tag', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 0, NULL, NULL, NULL, NULL, NULL, 6, NULL,
 'Find DNA corroboration'),

-- SINGLE_SOURCE_DEPENDENCE: blood relatives only — source corroboration not a goal for spouses
('SIGNAL_SINGLE_SOURCE_DEPENDENCE', 'evidence', 'integrity', 25,
 'single-source reliance', 'catches illusion-of-certainty problem', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 3, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find additional sources'),

-- UNSOURCED_FAMILY_EVENTS: blood relatives only — sourcing family events not a goal for spouses
-- Note: applies_always=TRUE in live table (not requires_family_events)
('SIGNAL_UNSOURCED_FAMILY_EVENTS', 'evidence', 'integrity', 25,
 'unsourced family events', 'marriages and child births without sources', 'evidence',
 TRUE,  NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Source family events'),

-- COMMON_ANCESTOR_UNVERIFIED: tag-based applicability handles correctly
('SIGNAL_COMMON_ANCESTOR_UNVERIFIED', 'evidence', 'integrity', 20,
 'common ancestor unverified', 'Common DNA Ancestor tag present but no DNA match paths through this person', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL,
 'Verify common ancestor'),

-- DNA_CITATION_MISSING: tag-based applicability handles correctly
('SIGNAL_DNA_CITATION_MISSING', 'evidence', 'integrity', 20,
 'DNA match citation missing', 'DNA Match tag present but no Ancestry citation in tree', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL,
 'Add DNA citation'),

-- TRANSCRIPT_ONLY_FACTS: blood relatives only — transcript pipeline only relevant for blood relatives
('SIGNAL_TRANSCRIPT_ONLY_FACTS', 'evidence', 'integrity', 20,
 'transcript facts not in tree', 'document contains facts not yet reflected in tree', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Verify transcript facts'),

-- UNCOVERED_SOURCES: blood relatives only — source coverage not a goal for spouses
-- Note: moved to evidence dimension in live table
('SIGNAL_UNCOVERED_SOURCES', 'evidence', 'integrity', 20,
 'uncovered cited sources', 'cited sources with no downloaded document', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Find source documents'),

-- LOW_EVIDENCE_DENSITY: blood relatives only — evidence quality not a goal for spouses
('SIGNAL_LOW_EVIDENCE_DENSITY', 'evidence', 'integrity', 15,
 'low source density (some unsourced facts)', 'avg sources per fact < 1.0', 'evidence',
 TRUE,  NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Add more sources'),

-- IMPRECISE_DATES: blood relatives only — date precision not a goal for spouses
('SIGNAL_IMPRECISE_DATES', 'evidence', 'integrity', 10,
 'imprecise dates', 'post-1837 civil registration events where a certificate should exist', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Confirm precise dates'),

-- INCOMPLETE_NAME: always applicable including spouses
('SIGNAL_INCOMPLETE_NAME', 'evidence', 'integrity', 10,
 'incomplete name', 'suggests exhaustive searches not done', 'evidence',
 TRUE,  NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
 'Complete name record'),

-- IMPRECISE_PLACES: blood relatives only — place precision not a goal for spouses
('SIGNAL_IMPRECISE_PLACES', 'evidence', 'integrity', 10,
 'imprecise places', 'post-1837 records should resolve to town/parish level', 'evidence',
 FALSE, 1600, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE,
 'Confirm precise places'),

-- ================================================================
-- NARRATIVE signals — all applies_always; not blood-restricted
-- ================================================================

('SIGNAL_MULTIPLE_SPOUSES',   'family',  'narrative', 40, 'multiple spouses',                NULL,                                    'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Research multiple marriages'),
('SIGNAL_MIGRANT',            'texture', 'narrative', 30, 'evidence of migration',           NULL,                                    'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Investigate migration'),
('SIGNAL_CONFIRMED_MILITARY', 'texture', 'narrative', 25, 'confirmed military service',      'has _MILT event or MilitaryRecord doc', 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Research military records'),
('SIGNAL_NEWSPAPER_MENTION',  'texture', 'narrative', 25, 'newspaper mention',               'rare: newspaper clipping filed',        'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Find newspaper records'),
('SIGNAL_TRANSCRIPT_RICH',    'context', 'narrative', 25, 'rich primary source material',    '>= 2 non-BMDIndex transcribed docs',    'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Review transcripts'),
('SIGNAL_YOUNG_DEATH',        'texture', 'narrative', 20, 'young death',                     NULL,                                    'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Research early death'),
('SIGNAL_VARIED_OCCUPATIONS', 'texture', 'narrative', 20, 'varied occupational history',     NULL,                                    'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Research occupations'),
('SIGNAL_WILL_OR_PROBATE',    'texture', 'narrative', 20, 'will or probate record',          'wealth/family dynamics',               'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Research will or probate'),
('SIGNAL_LARGE_FAMILY',       'texture', 'narrative', 15, 'large family (>= 6 children)',    'social history angle',                  'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Research family details'),
('SIGNAL_STORY_WRITTEN',      'texture', 'narrative',-50, 'story already written',           'suppresses narrative priority',         'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 'Story published');


In [0]:
%sql
CREATE OR REPLACE TABLE genealogy.gold_research_signal_action (
  signal_code   STRING,
  action_code   STRING,
  action_weight INT
);

INSERT INTO genealogy.gold_research_signal_action VALUES
  -- Evidence signals
  ('SIGNAL_NO_DOCUMENTS_AT_ALL',       'DOWNLOAD_DOCUMENTS',          3),
  ('SIGNAL_TRANSCRIPT_ONLY_FACTS',     'RECONCILE_TRANSCRIPT_FACTS',  3),
  ('SIGNAL_FACT_CONFLICT',             'RESOLVE_CONFLICTS',           3),
  ('SIGNAL_DOCS_NOT_TRANSCRIBED',      'TRANSCRIBE_DOCUMENTS',        3),
  ('SIGNAL_UNCOVERED_SOURCES',         'DOWNLOAD_DOCUMENTS',          3),
  ('SIGNAL_LOW_EVIDENCE_DENSITY',      'SEARCH_CENSUS',               2),
  ('SIGNAL_VERY_LOW_EVIDENCE_DENSITY', 'SEARCH_CENSUS',               2),
  ('SIGNAL_UNSOURCED_FAMILY_EVENTS',   'VERIFY_FAMILY_EVENTS',        3),
  ('SIGNAL_INCOMPLETE_NAME',           'SEARCH_MARRIAGE_RECORDS',     3),
  ('SIGNAL_INCOMPLETE_NAME',           'SEARCH_BIRTH_RECORDS',        2),
  ('SIGNAL_IMPRECISE_DATES',           'ORDER_BMD_CERTIFICATE',       3),
  ('SIGNAL_IMPRECISE_PLACES',          'ORDER_BMD_CERTIFICATE',       3),
  -- Completeness signals
  ('SIGNAL_NO_BIRTH_RECORDED',         'SEARCH_BIRTH_RECORDS',        3),
  ('SIGNAL_NO_DEATH_RECORDED',         'SEARCH_DEATH_RECORDS',        3),
  ('SIGNAL_NO_MARRIAGES',              'SEARCH_MARRIAGE_RECORDS',     3),
  ('SIGNAL_NO_CHILDREN',               'SEARCH_CENSUS',               3),
  ('SIGNAL_NO_CHILDREN',               'SEARCH_BIRTH_RECORDS',        2),
  ('SIGNAL_MISSING_CENSUS_COVERAGE',   'SEARCH_CENSUS',               3),
  ('SIGNAL_LATE_LIFE_GAP',             'INVESTIGATE_LATER_LIFE',      3),
  ('SIGNAL_LATE_LIFE_GAP',             'SEARCH_DEATH_RECORDS',        2),
  ('SIGNAL_EARLY_LIFE_ONLY',           'SEARCH_CENSUS',               3),
  ('SIGNAL_CHILD_GAPS',                'SEARCH_BIRTH_RECORDS',        3),
  ('SIGNAL_CHILD_GAPS',                'SEARCH_CENSUS',               2),
  ('SIGNAL_MISSING_BURIAL',            'SEARCH_BURIAL_RECORDS',       3),
  ('SIGNAL_MISSING_BURIAL',            'SEARCH_DEATH_RECORDS',        2),
  ('SIGNAL_NO_RESIDENCE',              'SEARCH_CENSUS',               3),
  ('SIGNAL_MISSING_PARENT',            'TRACE_PARENTS',               3),
  ('SIGNAL_MULTIPLE_SPOUSES',          'VERIFY_FAMILY_EVENTS',        3),
  ('SIGNAL_LARGE_FAMILY',              'TRACE_CHILDREN_FORWARD',      3),
  ('SIGNAL_LARGE_FAMILY',              'SEARCH_BIRTH_RECORDS',        2),
  ('SIGNAL_UNCONFIRMED_MILITARY',      'SEARCH_MILITARY_RECORDS',     3),
  ('SIGNAL_MISSING_OCCUPATION',        'SEARCH_CENSUS',               3),
  ('SIGNAL_MISSING_OCCUPATION',        'INVESTIGATE_OCCUPATION',      2),
  -- Narrative signals
  ('SIGNAL_CONFIRMED_MILITARY',        'INVESTIGATE_MILITARY',        3),
  ('SIGNAL_VARIED_OCCUPATIONS',        'INVESTIGATE_OCCUPATION',      3),
  ('SIGNAL_MIGRANT',                   'INVESTIGATE_MIGRATION',       3),
  ('SIGNAL_YOUNG_DEATH',               'SEARCH_DEATH_RECORDS',        3),
  ('SIGNAL_POSSIBLE_MARRIAGE',         'SEARCH_MARRIAGE_RECORDS',     3),
  ('SIGNAL_POSSIBLE_MARRIAGE',         'SEARCH_CENSUS',               2),
  ('SIGNAL_POSSIBLE_CHILDREN',         'SEARCH_BIRTH_RECORDS',        3),
  ('SIGNAL_POSSIBLE_CHILDREN',         'SEARCH_CENSUS',               2),
  ('SIGNAL_POSSIBLE_RESIDENCE',        'SEARCH_CENSUS',               3),
  -- DNA signals
  ('SIGNAL_DNA_CITATION_MISSING',      'INVESTIGATE_DNA_MATCH',       3),
  ('SIGNAL_DNA_PATH_INCOMPLETE',       'INVESTIGATE_DNA_MATCH',       3),
  ('SIGNAL_NO_DNA_CORROBORATION',      'INVESTIGATE_DNA_MATCH',       3),
  ('SIGNAL_COMMON_ANCESTOR_UNVERIFIED','INVESTIGATE_DNA_MATCH',       3);
  -- Intentionally no mapping (score but don't drive a closeable task):
  --   SIGNAL_STORY_WRITTEN, SIGNAL_TRANSCRIPT_RICH, SIGNAL_NEWSPAPER_MENTION,
  --   SIGNAL_WILL_OR_PROBATE, SIGNAL_VERY_LOW_EVIDENCE_DENSITY,
  --   SIGNAL_LOW_EVIDENCE_DENSITY, SIGNAL_SINGLE_SOURCE_DEPENDENCE


In [0]:
%sql
-- ── 1. Confirm requires_blood_relative column exists and is set correctly ────
-- Signals WITH requires_blood_relative = TRUE (expect 18)
SELECT signal_code, dimension, base_score, requires_blood_relative
FROM genealogy.ref_signal_weights
WHERE requires_blood_relative = TRUE
ORDER BY dimension, base_score DESC;


In [0]:
%sql
-- ── 2. Signals with no action mapping (expected — see comment in Cell 4) ────
SELECT w.signal_code, w.dimension, w.intent, w.base_score
FROM genealogy.ref_signal_weights w
LEFT JOIN genealogy.gold_research_signal_action a ON a.signal_code = w.signal_code
WHERE a.signal_code IS NULL
  AND w.signal_code NOT IN (
    'SIGNAL_STORY_WRITTEN',
    'SIGNAL_TRANSCRIPT_RICH',
    'SIGNAL_NEWSPAPER_MENTION',
    'SIGNAL_WILL_OR_PROBATE',
    'SIGNAL_VERY_LOW_EVIDENCE_DENSITY',
    'SIGNAL_LOW_EVIDENCE_DENSITY',
    'SIGNAL_SINGLE_SOURCE_DEPENDENCE'
  )
ORDER BY w.dimension, w.intent;

-- ── 3. Action mappings pointing to non-existent codes (expect 0 rows) ────────
SELECT sa.signal_code, sa.action_code
FROM genealogy.gold_research_signal_action sa
LEFT JOIN genealogy.gold_research_action a ON a.action_code = sa.action_code
WHERE a.action_code IS NULL;

-- ── 4. Total signal count and requires_blood_relative summary ─────────────────
SELECT
  COUNT(*) AS total_signals,
  SUM(CASE WHEN requires_blood_relative = TRUE THEN 1 ELSE 0 END) AS blood_restricted,
  SUM(CASE WHEN requires_blood_relative IS NULL THEN 1 ELSE 0 END) AS unrestricted,
  SUM(CASE WHEN outcome_label IS NOT NULL THEN 1 ELSE 0 END) AS has_outcome_label
FROM genealogy.ref_signal_weights;
